# 08 | Cross-Links: Portfolio Share, Spillovers, and Diminishing Returns

This notebook tests cross-technology spillovers, does spending on technology A
Granger-predict publications in technology B? We test 4 candidate pairs
motivated by physical/chemical adjacency.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import f as f_dist
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
panel = pd.read_csv("../data/processed/merged_panel.csv")
panel = panel.sort_values(["country", "technology", "year"])


## Country-level Hydrogen β vs portfolio share — does focus matter?

In [ ]:
# Country-level Hydrogen elasticity: regress Δpubs on Δspending within each country
def country_beta(df):
    df = df.sort_values("year").assign(
        d_pubs=lambda d: d["pub_count"].diff(),
        d_spend=lambda d: d["spending_usd_ppp_millions"].diff(),
    ).dropna()
    if len(df) < 8:
        return np.nan
    x = df["d_spend"].values; y = df["d_pubs"].values
    if np.std(x) == 0:
        return np.nan
    return np.cov(x, y, ddof=0)[0, 1] / np.var(x)

h = panel.query("technology == 'Hydrogen & fuel cells'")
betas = h.groupby("country").apply(country_beta).rename("hydrogen_beta")

shares = (
    panel.groupby(["country", "technology"])["spending_usd_ppp_millions"].sum()
         .groupby(level=0).apply(lambda s: s / s.sum())
         .unstack()["Hydrogen & fuel cells"]
         .rename("hydrogen_share")
)
share_beta = pd.concat([betas, shares], axis=1).dropna()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(share_beta["hydrogen_share"], share_beta["hydrogen_beta"], s=30)
for c, row in share_beta.iterrows():
    ax.annotate(c, (row["hydrogen_share"], row["hydrogen_beta"]), fontsize=7, alpha=0.7)
ax.axhline(0, color="grey", ls="--", lw=0.5)
ax.set_xlabel("Hydrogen share of public R&D portfolio")
ax.set_ylabel("Hydrogen Δpubs / Δspend slope")
ax.set_title("Country focus on hydrogen vs marginal research return")
plt.tight_layout()
plt.savefig("../figures/crosslink_share_vs_granger.png", dpi=150, bbox_inches="tight")
plt.show()


## Cross-technology spillover Granger tests (4 pairs)

In [ ]:
def diff_within(g):
    g = g.sort_values("year").copy()
    g["d_spending"] = g["spending_usd_ppp_millions"].diff()
    g["d_pubs"] = g["pub_count"].diff()
    return g

panel = panel.groupby(["country", "technology"], group_keys=False).apply(diff_within)


def cross_granger(spend_tech, pub_tech, lags=4):
    sp = panel.query("technology == @spend_tech")[["country", "year", "d_spending"]]
    pb = panel.query("technology == @pub_tech")[["country", "year", "d_pubs"]]
    df = sp.merge(pb, on=["country", "year"]).sort_values(["country", "year"]).dropna()

    def lagcols(col):
        return pd.concat({f"{col}_l{k}": df.groupby("country")[col].shift(k)
                          for k in range(1, lags + 1)}, axis=1)

    Y = df["d_pubs"]
    Xy = lagcols("d_pubs")
    Xx = lagcols("d_spending")
    cd = pd.get_dummies(df["country"], drop_first=True).astype(float)
    Xf = pd.concat([Xy, Xx, cd], axis=1)
    Xr = pd.concat([Xy, cd], axis=1)
    mask = pd.concat([Y, Xf], axis=1).dropna().index
    Y, Xf, Xr = Y.loc[mask], Xf.loc[mask], Xr.loc[mask]
    full = sm.OLS(Y, sm.add_constant(Xf)).fit()
    rest = sm.OLS(Y, sm.add_constant(Xr)).fit()
    F = ((rest.ssr - full.ssr) / lags) / (full.ssr / full.df_resid)
    p = 1 - f_dist.cdf(F, lags, full.df_resid)
    return F, p, int(full.nobs)


PAIRS = [
    ("Hydrogen & fuel cells", "CO2 capture & storage"),
    ("CO2 capture & storage", "Biofuels"),
    ("Solar", "Wind"),
    ("Nuclear", "Hydrogen & fuel cells"),
]

rows = []
for s, p in PAIRS:
    F, pval, n = cross_granger(s, p)
    rows.append({"spend_tech": s, "pub_tech": p, "F": F, "p": pval, "n": n})

cross = pd.DataFrame(rows)
cross.to_csv("../results/crosslink_results.csv", index=False)
cross


## Spillover time series — significant pairs

In [ ]:
def plot_spillover(spend_tech, pub_tech, fname):
    sp = (
        panel.query("technology == @spend_tech")
             .groupby("year")["spending_usd_ppp_millions"].sum()
             .rename("spend")
    )
    pb = (
        panel.query("technology == @pub_tech")
             .groupby("year")["pub_count"].sum()
             .rename("pubs")
    )
    df = pd.concat([sp, pb], axis=1).dropna()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax2 = ax.twinx()
    ax.plot(df.index, df["spend"], color="C0", label=f"{spend_tech} spending")
    ax2.plot(df.index, df["pubs"], color="C3", label=f"{pub_tech} pubs")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"{spend_tech} spending (USD M)", color="C0")
    ax2.set_ylabel(f"{pub_tech} pubs", color="C3")
    plt.title(f"Spillover: {spend_tech} → {pub_tech}")
    plt.tight_layout()
    plt.savefig(f"../figures/{fname}.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_spillover("Hydrogen & fuel cells", "CO2 capture & storage",
               "spillover_hydrogen_fuel_cells_to_co2_capture_storage")
plot_spillover("CO2 capture & storage", "Biofuels",
               "spillover_co2_capture_storage_to_biofuels")


## Output

- `../results/crosslink_results.csv`, F-stats and p-values for 4 spillover pairs.
- `../figures/crosslink_share_vs_granger.png`, two spillover plots.

**Findings:** A "carbon-management cluster" connects Hydrogen → CO2 capture
(F=20.9, p<1e-8) and CO2 capture → Biofuels (F=5.3, p≈0.005). The hypothesized
electrification spillover (Solar → Wind) and Nuclear → Hydrogen are not
detected. Spending on one technology in the cluster appears to partially fund
research in the others.
